In [2]:
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

base_dir = r'C:\Users\Admin\Downloads\Compressed\FaceForensics++\data_flat'

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
)

train_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

validation_generator = datagen.flow_from_directory(
    base_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)


Found 8453 images belonging to 2 classes.
Found 2112 images belonging to 2 classes.


In [4]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer=Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(train_generator, validation_data=validation_generator, epochs=15)

model.save('efficientnetb0.h5')


16705208/16705208 [==============================] - 6s 0us/step
Epoch 1/15
265/265 [==============================] - 230s 854ms/step - loss: 0.6950 - accuracy: 0.5261 - val_loss: 0.6915 - val_accuracy: 0.5426
Epoch 2/15
265/265 [==============================] - 235s 886ms/step - loss: 0.6921 - accuracy: 0.5393 - val_loss: 0.6900 - val_accuracy: 0.5426
Epoch 3/15
265/265 [==============================] - 649s 2s/step - loss: 0.6904 - accuracy: 0.5406 - val_loss: 0.6904 - val_accuracy: 0.5426
Epoch 4/15
265/265 [==============================] - 777s 3s/step - loss: 0.6908 - accuracy: 0.5431 - val_loss: 0.6895 - val_accuracy: 0.5426
Epoch 5/15
265/265 [==============================] - 750s 3s/step - loss: 0.6906 - accuracy: 0.5413 - val_loss: 0.6896 - val_accuracy: 0.5426
Epoch 6/15
265/265 [==============================] - 754s 3s/step - loss: 0.6907 - accuracy: 0.5424 - val_loss: 0.6896 - val_accuracy: 0.5426
Epoch 7/15
265/265 [==============================] - 768s 3s/step - lo

C:\Users\Admin\anaconda3\envs\deepfake_detection\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [6]:
# Evaluate the model on the validation data
val_loss, val_accuracy = model.evaluate(validation_generator)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

from sklearn.metrics import confusion_matrix, classification_report

# Reset the validation generator to ensure predictions start from the beginning
validation_generator.reset()

# Get predictions on the validation set
preds = model.predict(validation_generator)
predicted_classes = (preds > 0.5).astype(int).ravel()

# Get true class labels
true_classes = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

# Generate and print the confusion matrix
cm = confusion_matrix(true_classes, predicted_classes)
print("Confusion Matrix:")
print(cm)

# Generate and print the classification report
report = classification_report(true_classes, predicted_classes, target_names=class_labels)
print("Classification Report:")
print(report)



66/66 [==============================] - 154s 2s/step - loss: 0.6895 - accuracy: 0.5426
Validation Loss: 0.6895
Validation Accuracy: 0.5426
66/66 [==============================] - 158s 2s/step
Confusion Matrix:
[[   0  966]
 [   0 1146]]
Classification Report:
              precision    recall  f1-score   support

        fake       0.00      0.00      0.00       966
        real       0.54      1.00      0.70      1146

    accuracy                           0.54      2112
   macro avg       0.27      0.50      0.35      2112
weighted avg       0.29      0.54      0.38      2112



C:\Users\Admin\anaconda3\envs\deepfake_detection\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Admin\anaconda3\envs\deepfake_detection\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Admin\anaconda3\envs\deepfake_detection\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_sta